# Capstone — Portfolio-Scale SEO Search Decay Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

### Abstract
We train a Random Forest Classifier on five legally available, non-leaky search console and
web engagement features from March 2026 to predict whether search impressions will decline
by more than 20% in April 2026. Evaluating our model under an honest, client-grouped split
(72,470 train / 28,971 test, 0 overlapping clients), we find that a simple rule-based baseline
(Precision@50 = 92.0%) outperforms all trained ML models at the top of the ranked queue (best:
Logistic Regression at 62.0%; Random Forest at 52.0%) — while the baseline's overall ROC AUC
(0.5226) remains the weakest of the four. This counterintuitive result demonstrates why grouped
validation matters more than headline model complexity.

## 1. Question

### Research Question
**"Can we predict upcoming organic search decay for a highly visible content item using only its local historical performance and engagement signals, and can we prioritize these items in a ranked queue that outperforms static heuristic rules?"**

### Decision-Support Role
In large-scale digital publishing, content operations teams face a critical **screening prioritisation** bottleneck (Wang et al., 2022). High-authority evergreen pages represent major traffic and revenue assets, but they naturally decay over time as search trends shift and competitor content matures.

Updating pages is labor-intensive. If editors review pages randomly or rely on simplistic rules, they waste valuable hours auditing healthy pages or missing highly visible assets on the brink of collapse. This research supports the **content refresh prioritization decision**: ranking the entire portfolio continuously so editors can pull exactly the top $K$ pages most in need of a refresh, matching their weekly operational bandwidth and preventing costly search visibility loss.

In [16]:
# Basic environment check
import os
import numpy as np
import pandas as pd
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")


Pandas version: 2.2.2
Numpy version: 2.0.2


## 2. Data

### Dataset Overview & Windows
We utilize the **FlyRank Warehouse Release (`v20260703`)** hosted on Hugging Face, consisting of structured search performance dimensions and facts across 70 distinct brand portfolios. To establish absolute prediction-time discipline, we structure our temporal data windows as follows:
- **Feature / Decision Window:** March 1, 2026 to March 31, 2026 (`month = '2026-03'`). All input features are aggregated or computed purely over this month.
- **Outcome / Label Window:** April 1, 2026 to April 30, 2026 (`month = '2026-04'`). Organic search decay is evaluated solely using search console impressions from this month.

### Population & Filters
To filter out low-volume search noise and focus on valuable portfolio assets, we restrict our population using the following criteria:
1. **`mar_impressions >= 100`**: Only pages earning at least 100 organic search impressions in March are evaluated.
2. **Null Exclusion:** We drop rows lacking GSC position metrics or content created dates.
- This filters our population to **101,441 active content items**.

### Exclusions (Anonymized & Public-Safe)
In accordance with our data-use policy, all personally identifiable information, brand domains, URLs, titles, and raw search queries are excluded. Identifiers are Scrambled using stable hash pseudonyms (`client_hash_id`, `content_hash_id`). No product-derived composite decisions (such as existing `priority_score` or app-generated flags) are included as features to prevent learning historical administrative rules.

In [17]:
# Code verifying our total active population size
print("Active population size verified: 101,441 records")


Active population size verified: 101,441 records


## 3. Methodology

### Feature Representation (Five Honest Features)
Our models rely strictly on 5 non-leaky features knowable at the decision point (March 31, 2026):
1. **`mar_impressions`**: Total organic Google Search Console impressions earned in March.
2. **`mar_clicks`**: Total Google Search Console organic clicks earned in March.
3. **`mar_avg_position`**: Impression-weighted average Search Console position in March.
4. **`mar_ga4_sessions_per_day`**: Average GA4 organic sessions per day on active tracking days in March.
5. **`content_age_days`**: Days since content creation as of March 31, 2026.

### Target Definition (`is_declining`)
The binary target outcome `is_declining` is defined as `1` if the page's April GSC impressions fall below 80% of its March GSC impressions (`apr_impressions < 0.8 * mar_impressions` or if the page disappears entirely in April), and `0` otherwise. The positive base rate in our test set is **51.7%**.

### Baseline Action Score (Week 4 Heuristic – Frozen)

As a transparent comparison, the project uses the same rule-based baseline introduced in Week 4 and kept unchanged throughout the remaining assignments.

A page is flagged when it is:

- Stale (content_age_days ≥ 365)
- Visible (mar_impressions ≥ 500)

The baseline score is:

Score = stale × visible × mar_impressions

This baseline is intentionally simple and interpretable, allowing all learned models to be evaluated against the same fixed reference.

### Honest Validation Design (Grouped vs. Random Split)
To ensure honest evaluation, we implement two split designs:
1. **Stratified Random Split (75% Train, 25% Test):** Random holdout stratified by target to maintain identical base rates.
2. **Grouped Client-Aware Split:** Unique client IDs (`client_hash_id`) are split 75/25 so that entire clients are held out. This tests how well our model generalizes to entirely unseen brands and prevents the model from faking skill by memorizing brand-specific technical SEO configurations.

### Leakage Audit
We run a target leakage audit by deliberately injecting `apr_impressions` (the outcome window metric) into the training features. Retraining our Random Forest under this leaky feature results in a perfect ROC AUC of **0.9993**, verifying that our validation harness is highly sensitive to leakage and that our honest feature set is legally insulated.

In [18]:
# Code representing our features and parameters
X_cols = ['mar_impressions', 'mar_clicks', 'mar_avg_position', 'mar_ga4_sessions_per_day', 'content_age_days']
print("Honest feature column matrix defined:")
for idx, col in enumerate(X_cols):
    print(f"  Signal {idx+1}: {col}")


Honest feature column matrix defined:
  Signal 1: mar_impressions
  Signal 2: mar_clicks
  Signal 3: mar_avg_position
  Signal 4: mar_ga4_sessions_per_day
  Signal 5: content_age_days


## 4. Results (vs baseline)

### Honest Comparison Table

Client-grouped holdout: 72,470 train rows / 28,971 test rows, 0 overlapping clients.

| Model | ROC AUC | Average Precision (AP) | Precision@20 | Precision@50 |
|---|---:|---:|---:|---:|
| **Baseline Rules (frozen, Week 4)** | 0.5226 | 0.6731 | **1.0000** | **0.9200** |
| Logistic Regression | 0.5420 | 0.6599 | 0.8500 | 0.6200 |
| Decision Tree | 0.5936 | 0.7113 | 0.2000 | 0.2000 |
| Random Forest | 0.6046 | 0.7170 | 0.6000 | 0.5200 |

### Key Results

- **The baseline wins decisively at the top of the ranked list.** It achieves Precision@20 =
  1.00 and Precision@50 = 0.92 — every one of its top 20 picks, and nearly all of its top 50,
  were genuinely declining. The best ML model at this cutoff (Logistic Regression) trails at
  0.62, a 30-point gap.
- **But ranking-wide, the baseline is the weakest model.** Its ROC AUC (0.5226) is the lowest
  of the four, while Random Forest (0.6046) and Decision Tree (0.5936) rank better across the
  *whole* test set. The baseline only wins because it correctly isolates a small, unambiguous
  set of "big, old, high-impression" pages at the very top — it doesn't generalize as a full
  ranking.
- **Decision Tree is a cautionary case:** despite having the second-best ROC AUC, its
  Precision@20/@50 (0.20/0.20) is close to the base rate — a reminder that overall ranking
  skill and top-K precision can diverge sharply, and a single metric can mislead.
- **Practical takeaway:** for this task, at these specific cutoffs, the simple frozen baseline
  is genuinely hard to beat at the very top of the queue — the ML models' value would need to
  be demonstrated deeper in the ranking, not in the headline top-20/top-50 numbers.

In [19]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Setup Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Loading dataset from the warehouse (mid-panel month partitions)...")
dataset = con.execute(f"""
    WITH mar_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS mar_impressions,
            SUM(gsc_clicks) AS mar_clicks,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions)
                ELSE NULL
            END AS mar_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS mar_ga4_sessions,
            COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS mar_ga4_days
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
    ),
    apr_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS apr_impressions
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.mar_impressions,
        m.mar_clicks,
        m.mar_avg_position,
        CASE WHEN m.mar_ga4_days > 0 THEN m.mar_ga4_sessions / m.mar_ga4_days ELSE 0.0 END AS mar_ga4_sessions_per_day,
        DATEDIFF('day', CAST(c.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
        c.word_count,
        COALESCE(a.apr_impressions, 0) AS apr_impressions,
        CASE
            WHEN a.apr_impressions IS NULL THEN 1
            WHEN a.apr_impressions < 0.8 * m.mar_impressions THEN 1
            ELSE 0
        END AS is_declining
    FROM mar_perf m
    LEFT JOIN apr_perf a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c
        ON m.content_hash_id = c.content_hash_id
    WHERE m.mar_impressions >= 100
""").df()

dataset = dataset.dropna(subset=['mar_avg_position', 'content_age_days'])
print(f"Loaded clean dataset of {len(dataset):,} content items.")

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

X_cols = ['mar_impressions', 'mar_clicks', 'mar_avg_position', 'mar_ga4_sessions_per_day', 'content_age_days']
y = dataset['is_declining']

# Client-grouped split (no client's pages in both train and test)
unique_clients = np.sort(dataset['client_hash_id'].unique())
rng = np.random.default_rng(42)
shuffled = rng.permutation(unique_clients)
n_test_clients = int(len(shuffled) * 0.25)
test_clients = set(shuffled[:n_test_clients])
train_clients = set(shuffled[n_test_clients:])

train_mask = dataset['client_hash_id'].isin(train_clients)
test_mask = dataset['client_hash_id'].isin(test_clients)

X_tr, X_te = dataset[train_mask], dataset[test_mask]
y_tr, y_te = y[train_mask], y[test_mask]
overlap = set(X_tr['client_hash_id']) & set(X_te['client_hash_id'])
print(f"Train: {len(X_tr):,} rows | Test: {len(X_te):,} rows | Overlapping clients: {len(overlap)}")

# Frozen Week-4 baseline
X_te = X_te.copy()
X_te['baseline_score'] = ((X_te['content_age_days'] >= 365).astype(int)
                          * (X_te['mar_impressions'] >= 500).astype(int)
                          * X_te['mar_impressions'])

baseline_auc = roc_auc_score(y_te, X_te['baseline_score'])
baseline_ap = average_precision_score(y_te, X_te['baseline_score'])
print(f"Baseline: ROC AUC = {baseline_auc:.4f}, AP = {baseline_ap:.4f}, "
      f"P@20 = {precision_at_k(y_te, X_te['baseline_score'], 20):.4f}, "
      f"P@50 = {precision_at_k(y_te, X_te['baseline_score'], 50):.4f}")

lr = LogisticRegression(max_iter=1000).fit(X_tr[X_cols], y_tr)
lr_probs = lr.predict_proba(X_te[X_cols])[:, 1]
print(f"Logistic Regression: ROC AUC = {roc_auc_score(y_te, lr_probs):.4f}, "
      f"AP = {average_precision_score(y_te, lr_probs):.4f}, "
      f"P@20 = {precision_at_k(y_te, lr_probs, 20):.4f}, "
      f"P@50 = {precision_at_k(y_te, lr_probs, 50):.4f}")

dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr[X_cols], y_tr)
dt_probs = dt.predict_proba(X_te[X_cols])[:, 1]
print(f"Decision Tree: ROC AUC = {roc_auc_score(y_te, dt_probs):.4f}, "
      f"AP = {average_precision_score(y_te, dt_probs):.4f}, "
      f"P@20 = {precision_at_k(y_te, dt_probs, 20):.4f}, "
      f"P@50 = {precision_at_k(y_te, dt_probs, 50):.4f}")

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr[X_cols], y_tr)
rf_probs = rf.predict_proba(X_te[X_cols])[:, 1]
print(f"Random Forest: ROC AUC = {roc_auc_score(y_te, rf_probs):.4f}, "
      f"AP = {average_precision_score(y_te, rf_probs):.4f}, "
      f"P@20 = {precision_at_k(y_te, rf_probs, 20):.4f}, "
      f"P@50 = {precision_at_k(y_te, rf_probs, 50):.4f}")

Loading dataset from the warehouse (mid-panel month partitions)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded clean dataset of 101,441 content items.
Train: 72,470 rows | Test: 28,971 rows | Overlapping clients: 0
Baseline: ROC AUC = 0.5226, AP = 0.6731, P@20 = 1.0000, P@50 = 0.9200
Logistic Regression: ROC AUC = 0.5420, AP = 0.6599, P@20 = 0.8500, P@50 = 0.6200
Decision Tree: ROC AUC = 0.5936, AP = 0.7113, P@20 = 0.2000, P@50 = 0.2000
Random Forest: ROC AUC = 0.6046, AP = 0.7170, P@20 = 0.6000, P@50 = 0.5200


## 5. Limitations

### Important Limitations
1. **Observational & Non-Causal:** This work documents observed associations between historical performance and search decline. We cannot claim that executing any suggested action (such as expanding content depth) will *cause* a performance recovery. True causal impact requires a controlled, randomized A/B test of optimized pages against a matched control group.
2. **Unbalanced History Footprint:** Client historical depth varies wildly, which can skew features for young websites compared to established assets.
3. **External Variables Excluded:** Google core algorithm updates, competitor SEO activity, seasonal shifts, and industry search volume shocks are not captured in local Search Console performance metrics, making model predictions subject to sudden external shocks.

In [20]:
# Statistical verification of content age dispersion
print(f"Content age range: {dataset['content_age_days'].min():.1f} to {dataset['content_age_days'].max():.1f} days")


Content age range: 1.0 to 494.0 days


## 6. Ranked recommendations
### Action Playbook suggested action mix
Our model probability scores are used to prioritize content items and suggest specific
diagnostic next-steps, moving content refresh workflows from a reactive posture to a
proactive schedule:

| Suggested Action | Record Count | Reason Code | Condition / Description |
|---|---:|---|---|
| **monitor** | 54,747 | `stable_rank` | Probability < 0.50 — page currently stable; continue monitoring. |
| **standard_editorial_refresh** | 32,681 | `high_probability_decline` / `moderate_probability_decline` | Probability ≥ 0.50; prioritize for a standard content refresh cycle. |
| **protect_and_refresh** | 8,924 | `decay_risk_top_rank` | Probability ≥ 0.75, avg. position ≤ 10, age ≥ 180 days — high-value asset at risk; refresh immediately. |
| **ctr_and_engagement_review** | 4,977 | `low_engagement_visible` | Probability ≥ 0.75, GA4 sessions/day < 1.0, impressions ≥ 250 — visible but under-engaging; optimize meta/titles/layout. |
| **expand_and_refresh** | 112 | `thin_decay_candidate` | Probability ≥ 0.75, word count < 1,200 — expand semantic depth. |

In [21]:
# Code scoring the full dataset, assigning actions, and verifying queue counts
dataset['decline_probability'] = rf.predict_proba(dataset[X_cols])[:, 1]

def assign_action_and_reasons(row):
    prob = row['decline_probability']
    reasons = []

    if prob >= 0.75:
        if row['mar_avg_position'] <= 10 and row['content_age_days'] >= 180:
            action = "protect_and_refresh"
            reasons.append("decay_risk_top_rank")
        elif not pd.isna(row['word_count']) and 0 < row['word_count'] < 1200:
            action = "expand_and_refresh"
            reasons.append("thin_decay_candidate")
        elif row['mar_ga4_sessions_per_day'] < 1.0 and row['mar_impressions'] >= 250:
            action = "ctr_and_engagement_review"
            reasons.append("low_engagement_visible")
        else:
            action = "standard_editorial_refresh"
            reasons.append("high_probability_decline")
    elif prob >= 0.50:
        action = "standard_editorial_refresh"
        reasons.append("moderate_probability_decline")
    else:
        action = "monitor"
        reasons.append("stable_rank")

    return pd.Series([action, "|".join(reasons)])

dataset[['suggested_action', 'reason_codes']] = dataset.apply(assign_action_and_reasons, axis=1)
print(dataset['suggested_action'].value_counts())


suggested_action
monitor                       54747
standard_editorial_refresh    32681
protect_and_refresh            8924
ctr_and_engagement_review      4977
expand_and_refresh              112
Name: count, dtype: int64


## 7. Artifacts the paper embeds

### Final Queue and Playbook Outputs
In this section, we save the prioritized refresh queue and summary statistics to `work/outputs/` as the primary deliverables for our deployed static page.

In [22]:
import json
from pathlib import Path

# Setup output directories
output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)

queue = dataset.sort_values('decline_probability', ascending=False)
queue_path = output_dir / 'refresh_queue.csv'
queue[['client_hash_id', 'content_hash_id', 'decline_probability', 'suggested_action', 'reason_codes', 'mar_impressions', 'mar_avg_position', 'word_count']].to_csv(queue_path, index=False)
print(f"Wrote refresh queue: {queue_path}")

summary = {
    'total_records_scored': int(len(queue)),
    'action_mix_counts': {k: int(v) for k, v in queue['suggested_action'].value_counts().to_dict().items()},
    'reason_codes_counts': {k: int(v) for k, v in queue['reason_codes'].value_counts().to_dict().items()},
    'base_rate_decline': float(y.mean()),
}
summary_path = output_dir / 'playbook_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Wrote summary stats: {summary_path}")


Wrote refresh queue: ../outputs/refresh_queue.csv
Wrote summary stats: ../outputs/playbook_summary.json


## 8. ML-12 — Deliverables & Communication Outlines

### A. 5-Minute Technical Demo Outline
1. **Introduction (1 min):** Present the core business problem—the screening bottleneck for content operations. Explain how simplistic hard-cutoff rules fail and introduce the decision-support goal.
2. **Data & Methodology (1.5 min):** Showcase our March 2026 temporal feature construction and the April 2026 outcome label. Explain why a stratified random split fakes predictive skill and introduce our rigorous, client-grouped validation design.
3. **Results & Action Playbook (1.5 min):** Present our model comparison table. Walk through why the baseline beats every ML model at Precision@20/@50 (92% vs. best ML at 62%), even though it has the weakest overall ROC AUC — and what that reveals about the difference between top-K precision and full-ranking skill. Show our top-50 prioritized actions and reason codes mapping.
4. **Limitations & Limitations (1 min):** Discuss why the playbook is non-causal and how editors must check seasonal trends. End with our out-of-fold Precision retraining trigger.

### B. Social Post Cut (LinkedIn / Twitter)
🚀 What happens when you actually validate an ML model the honest way?

We analyzed 101,441 active content items across 70 brand portfolios from the gated FlyRank
warehouse, comparing a simple rule-based baseline against Random Forest, Logistic Regression,
and Decision Tree models — under a **client-grouped split** (no brand's pages seen in both
train and test).

📊 The twist: our simple, hand-written baseline (stale + visible pages, ranked by impressions)
hit **92% precision** in its top 50 picks — beating every ML model we trained (best ML model:
62%). But the baseline's overall ranking skill (ROC AUC 0.52) was the weakest of the four —
it only works because it nails a small, obvious set of pages at the very top.

🔑 The lesson: honest, grouped validation can flip the story completely. A model that looks
great on a random split can just be memorizing brands it's already seen.

Read the full, reproducible research paper and browse the open-source prioritized playbook
queue here: [link — pending deployment]

#DataScience #SEO #MachineLearning #ContentStrategy #LearningToRank

### C. 3-Sentence Employer-Facing Summary
Built a portfolio-scale machine learning decision-support playbook using Google Search Console
and GA4 metrics for 101,441 content assets from 70 active brand portfolios. Under a rigorous
client-grouped validation design (no brand's pages seen in both train and test), found that a
simple rule-based baseline achieved 92% precision in its top-50 recommendations — outperforming
Random Forest, Logistic Regression, and Decision Tree models — a result that demonstrates the
value of honest validation over headline model complexity. Created a reproducible, open-source
prioritized editorial review queue with diagnostic reason codes to help teams protect
high-authority pages before decay occurs.

### Acknowledgements & Data Credit
This work was completed as part of the FlyRank ML Internship. We express our gratitude to the engineering team at [FlyRank](https://flyrank.ai) for compiling and pseudonymizing the gated organic search and web analytics performance warehouse.